# Sensus Penduduk Cleaning & Integration
**Tugas**: Menggabungkan data populasi tahunan (BPS) ke dataset harian ISPU berdasarkan Tahun dan Stasiun.

**Dataset yang digunakan:**
1. `Sensus-penduduk.csv`: Data populasi tahunan.
2. `merged_data_v2.csv`: Data harian ISPU + Hari Libur.

## 1. Load Libraries & Setup Paths

In [3]:
import pandas as pd
import os

population_path = '../dataset/jumlah-penduduk/Sensus-penduduk.csv'
main_data_path = 'dataset/merged_data_v2.csv'
output_path = 'dataset/merged_data_v3_population.csv'

## 2. Process Population Data (BPS)
Membaca data populasi, melakukan unpivot (wide-to-long), dan mapping kolom Kota ke kode Stasiun (DKI1-DKI5).

In [4]:
if os.path.exists(population_path):
    df_pop = pd.read_csv(population_path)
    
    # Melt: [Tahun, JAKARTA PUSAT...] -> [Tahun, Kota, Total_Penduduk]
    id_vars = [c for c in df_pop.columns if 'tahun' in c.lower()]
    df_pop = df_pop.melt(id_vars=id_vars, var_name='Kota', value_name='Total_Penduduk')
    df_pop = df_pop.rename(columns={id_vars[0]: 'tahun'})
    
    # Mapping Station
    station_map = {
        'JAKARTA PUSAT': 'DKI1', 'JAKARTA UTARA': 'DKI2', 
        'JAKARTA SELATAN': 'DKI3', 'JAKARTA TIMUR': 'DKI4', 'JAKARTA BARAT': 'DKI5'
    }
    df_pop['stasiun'] = df_pop['Kota'].str.upper().str.strip().map(station_map)
    df_pop_clean = df_pop.dropna(subset=['stasiun'])[['tahun', 'stasiun', 'Total_Penduduk']].copy()
    
    print(df_pop_clean.head())
else:
    print("Population file missing.")

   tahun stasiun Total_Penduduk
0   2010    DKI1         902973
1   2011    DKI1         906752
2   2012    DKI1         908829
3   2013    DKI1         906601
4   2014    DKI1         910381


## 3. Merge with Main Dateset
Menggabungkan dataset utama harian (`merged_data_v2.csv`) dengan data populasi berdasarkan `tahun` dan `stasiun`.

In [5]:
if os.path.exists(main_data_path):
    df_main = pd.read_csv(main_data_path)
    
    # Extract Year from Date
    df_main['tanggal'] = pd.to_datetime(df_main['tanggal'])
    df_main['tahun'] = df_main['tanggal'].dt.year
    
    # Merge (Left Join)
    df_final = pd.merge(df_main, df_pop_clean, on=['tahun', 'stasiun'], how='left')
    
    print(df_final[['tanggal', 'stasiun', 'tahun', 'Total_Penduduk']].head())
else:
    print("Main dataset missing.")

     tanggal stasiun   tahun Total_Penduduk
0 2010-01-01    DKI1  2010.0         902973
1 2010-01-02    DKI1  2010.0         902973
2 2010-01-03    DKI1  2010.0         902973
3 2010-01-04    DKI1  2010.0         902973
4 2010-01-05    DKI1  2010.0         902973


## 4. Save Output
Menyimpan hasil akhir ke `converted_merged_data_v3_population.csv`.

In [6]:
if 'df_final' in locals():
    df_final.to_csv(output_path, index=False)
    print(f"Data saved to {output_path}")
    print(f"Shape: {df_final.shape}")

Data saved to dataset/merged_data_v3_population.csv
Shape: (15412, 44)
